# Task 4 — Machine Translation with Seq2Seq Models

### Introduction
In this notebook, we implement a sequence-to-sequence Encoder-Decoder architecture in PyTorch for Machine Translation. The model is trained to translate short French sentences into English.

### Architecture Mechanics
- **Encoder**: An RNN/GRU that consumes the input sequence and produces a final context vector.
- **Decoder**: An RNN/GRU initialized with the Encoder's context vector, generating target tokens step-by-step.
- **Teacher Forcing**: During training, we feed the actual target word as the next input to the decoder instead of the decoder's prediction, which accelerates training convergence.



### Step 1 — Prepare Bilingual Sentence Pairs Dataset


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random

# Short English-French toy dataset
data = [
    ("i am tired", "je suis fatigué"),
    ("he is small", "il est petit"),
    ("she is happy", "elle est heureuse"),
    ("nous sommes explorateurs", "we are explorers"),
    ("they are fast", "ils sont rapides"),
    ("i am happy", "je suis heureux"),
    ("she is small", "elle est petite"),
    ("he is happy", "il est heureux")
]

# Build vocabulary dictionaries
def build_vocab(sentences):
    words = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
    for s in sentences:
        words.extend(s.lower().split())
    unique = sorted(list(set(words)))
    w2idx = {w: idx for idx, w in enumerate(unique)}
    idx2w = {idx: w for w, idx in w2idx.items()}
    return w2idx, idx2w

src_w2idx, src_idx2w = build_vocab([pair[1] for pair in data])
tgt_w2idx, tgt_idx2w = build_vocab([pair[0] for pair in data])

src_vocab_size = len(src_w2idx)
tgt_vocab_size = len(tgt_w2idx)

max_len = 6

def sentence_to_indices(sentence, w2idx):
    indices = [w2idx.get(w, w2idx["<UNK>"]) for w in sentence.lower().split()]
    indices.append(w2idx["<EOS>"])
    # Pad to max_len
    if len(indices) < max_len:
        indices += [w2idx["<PAD>"]] * (max_len - len(indices))
    return indices[:max_len]

# Create datasets
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.pairs = []
        for eng, fre in data:
            src = sentence_to_indices(fre, src_w2idx)
            tgt = sentence_to_indices(eng, tgt_w2idx)
            self.pairs.append((torch.tensor(src, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)))
            
    def __len__(self):
        return len(self.pairs)
        
    def __getitem__(self, idx):
        return self.pairs[idx]

dataset = TranslationDataset(data)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f"Source (French) Vocab Size: {src_vocab_size}")
print(f"Target (English) Vocab Size: {tgt_vocab_size}")



Source (French) Vocab Size: 20
Target (English) Vocab Size: 18


### Step 2 — Define Encoder-Decoder Seq2Seq Model


In [2]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, h = self.gru(embedded)
        # Return final hidden state as context vector
        return h

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, h):
        # x is single token index (batch, 1)
        embedded = self.embedding(x)
        out, h = self.gru(embedded, h)
        logits = self.fc(out)
        return logits, h

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        
        # Encode source sequence
        context = self.encoder(src)
        
        # Initialize decoder inputs with <SOS>
        dec_input = torch.ones(batch_size, 1, dtype=torch.long) * tgt_w2idx["<SOS>"]
        dec_hidden = context
        
        outputs = []
        for t in range(tgt_len):
            logits, dec_hidden = self.decoder(dec_input, dec_hidden)
            outputs.append(logits)
            
            # Predict top token
            top1 = logits.argmax(-1)
            
            # Decide to use Teacher Forcing or predicted token
            teacher_force = random.random() < teacher_forcing_ratio
            dec_input = tgt[:, t].unsqueeze(1) if teacher_force else top1
            
        return torch.cat(outputs, dim=1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = Encoder(src_vocab_size, embed_dim=16, hidden_dim=16)
decoder = Decoder(tgt_vocab_size, embed_dim=16, hidden_dim=16)
model = Seq2Seq(encoder, decoder).to(device)
print(model)



Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(20, 16, padding_idx=0)
    (gru): GRU(16, 16, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(18, 16, padding_idx=0)
    (gru): GRU(16, 16, batch_first=True)
    (fc): Linear(in_features=16, out_features=18, bias=True)
  )
)


### Step 3 — Train Model & Verify Translation


In [3]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 100
model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        logits = model(src, tgt, teacher_forcing_ratio=0.6)
        
        # Reshape for cross entropy
        loss = criterion(logits.view(-1, tgt_vocab_size), tgt.view(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_loss/len(loader):.4f}")

# Evaluation Function
def evaluate_translation(sentence):
    model.eval()
    with torch.no_grad():
        indices = sentence_to_indices(sentence, src_w2idx)
        in_tensor = torch.tensor([indices], dtype=torch.long).to(device)
        context = model.encoder(in_tensor)
        
        dec_input = torch.ones(1, 1, dtype=torch.long).to(device) * tgt_w2idx["<SOS>"]
        dec_hidden = context
        
        translated_words = []
        for _ in range(max_len):
            logits, dec_hidden = model.decoder(dec_input, dec_hidden)
            top1 = logits.argmax(-1).item()
            if top1 == tgt_w2idx["<EOS>"]:
                break
            translated_words.append(tgt_idx2w[top1])
            dec_input = torch.tensor([[top1]], dtype=torch.long).to(device)
            
    print(f"Input: '{sentence}'")
    print(f"Translation: '{' '.join(translated_words)}'")

print()
evaluate_translation("je suis fatigué")
evaluate_translation("elle est heureuse")



Epoch 20/100 - Train Loss: 0.4453


Epoch 40/100 - Train Loss: 0.1428


Epoch 60/100 - Train Loss: 0.0707


Epoch 80/100 - Train Loss: 0.0363


Epoch 100/100 - Train Loss: 0.0140

Input: 'je suis fatigué'
Translation: 'i am tired <PAD> <PAD> <PAD>'
Input: 'elle est heureuse'
Translation: 'she is happy <PAD> <PAD> <PAD>'


### Step 4 — Interactive Translation Interface
Type a French sentence or select one of the training sentences from the dropdown to test translation to English!


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

dropdown = widgets.Dropdown(
    options=[pair[1] for pair in data] + ["Custom Sentence..."],
    value="je suis fatigué",
    description="Select/Type:",
    layout=widgets.Layout(width="50%")
)

custom_input = widgets.Text(
    value="je suis fatigué",
    placeholder="Type a French sentence...",
    description="French:",
    layout=widgets.Layout(width="50%"),
    disabled=True
)

translate_btn = widgets.Button(
    description="Translate",
    button_style="info",
    icon="language",
    layout=widgets.Layout(width="20%")
)

output_area = widgets.Output()

def on_dropdown_change(change):
    if change['new'] == "Custom Sentence...":
        custom_input.disabled = False
    else:
        custom_input.value = change['new']
        custom_input.disabled = True

dropdown.observe(on_dropdown_change, names='value')

def on_translate(b):
    with output_area:
        clear_output()
        sentence = custom_input.value.strip()
        if not sentence:
            print("Please enter a sentence.")
            return
        
        evaluate_translation(sentence)

translate_btn.on_click(on_translate)

ui = widgets.VBox([
    widgets.HTML("<h3>Interactive French-to-English Translator</h3>"),
    dropdown,
    custom_input,
    translate_btn,
    output_area
])
display(ui)
on_translate(None)

